# 01 - Synthetic Netflix Generator Prototype

Plug-and-play first: use SDV single-table synthesizers on one row per rating, then run the same privacy/utility checks as before. The fastest baseline is `GaussianCopulaSynthesizer`; the stronger next options are `CTGANSynthesizer` and `TVAESynthesizer`.

Core references: SDV docs, Gaussian copulas, CTGAN, TVAE, Private-PGM/MST, and the Netflix de-anonymization paper are cited in `notebooks/README.md`.

In [ ]:
from __future__ import annotations

import os
import sys
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib-cache"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except ImportError:
    sns = None

try:
    from sdv.metadata import SingleTableMetadata
    from sdv.single_table import GaussianCopulaSynthesizer, CTGANSynthesizer, TVAESynthesizer
except ImportError as exc:
    raise ImportError("Install notebook ML deps with: python -m pip install -e '.[ml]'") from exc

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from guardrails_sensitive_data.anonymization import add_time_and_rating_features, evaluate_releases
from guardrails_sensitive_data.netflix_io import read_netflix_ratings
from guardrails_sensitive_data.recommender import fit_bias_recommender, mae, rmse, train_test_split_ratings

DATA_DIR = ROOT / "data" / "netflix"
SEED = 333
MAX_ROWS = 250_000
if sns is not None:
    sns.set_theme(style="whitegrid")

In [ ]:
ratings = read_netflix_ratings(DATA_DIR, max_rows=MAX_ROWS, include_date=True)
ratings = add_time_and_rating_features(ratings)
ratings["synthetic_row_id"] = np.arange(len(ratings), dtype=np.int64)
ratings.head()

## SDV baseline

`GaussianCopulaSynthesizer` is the shortest useful baseline. It learns marginal distributions plus covariance after reversible data transforms. Swap `SYNTHESIZER` to `"ctgan"` or `"tvae"` when you want a heavier neural model.

In [ ]:
SYNTHESIZER = "gaussian_copula"  # "ctgan" or "tvae" for heavier experiments
TRAIN_ROWS = min(75_000, len(ratings))
SAMPLE_ROWS = TRAIN_ROWS

sdv_train = ratings[["synthetic_row_id", "movie_id", "rating", "month"]].sample(TRAIN_ROWS, random_state=SEED)
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(sdv_train)
metadata.set_primary_key("synthetic_row_id")
for col in ["movie_id", "rating", "month"]:
    metadata.update_column(col, sdtype="categorical")

factory = {
    "gaussian_copula": lambda: GaussianCopulaSynthesizer(metadata, enforce_rounding=True),
    "ctgan": lambda: CTGANSynthesizer(metadata, epochs=50, verbose=True, enforce_rounding=True),
    "tvae": lambda: TVAESynthesizer(metadata, epochs=50, enforce_min_max_values=True, enforce_rounding=True),
}
synthesizer = factory[SYNTHESIZER]()
synthesizer.fit(sdv_train)
raw_synthetic = synthesizer.sample(num_rows=SAMPLE_ROWS)
raw_synthetic.head()

In [ ]:
def postprocess_sdv_ratings(frame: pd.DataFrame, real: pd.DataFrame, seed: int = SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    out = frame.copy()
    valid_movies = set(real["movie_id"].astype(int).unique())
    out["movie_id"] = pd.to_numeric(out["movie_id"], errors="coerce").round()
    out = out.dropna(subset=["movie_id"])
    out["movie_id"] = out["movie_id"].astype(int)
    out = out[out["movie_id"].isin(valid_movies)]
    out["rating"] = pd.to_numeric(out["rating"], errors="coerce").round().clip(1, 5)
    out = out.dropna(subset=["movie_id", "rating", "month"])
    out["rating"] = out["rating"].astype("int8")

    lengths = real.groupby("customer_id").size().clip(upper=150).to_numpy()
    synthetic_users = []
    cursor = 0
    while cursor < len(out):
        history_len = int(rng.choice(lengths))
        synthetic_users.extend([900_000_000 + len(synthetic_users)] * min(history_len, len(out) - cursor))
        cursor += history_len
    out = out.iloc[: len(synthetic_users)].copy()
    out["customer_id"] = synthetic_users
    out["date"] = out["month"].astype(str).where(out["month"].astype(str).eq("unknown"), out["month"].astype(str) + "-15")
    return add_time_and_rating_features(out[["movie_id", "customer_id", "rating", "date"]])

synthetic = postprocess_sdv_ratings(raw_synthetic, ratings)
synthetic.head()

## Quality, privacy, and utility checks

Keep the checks model-agnostic so Gaussian Copula, CTGAN, TVAE, or a future Private-PGM/MST generator can be compared with the same report.

In [ ]:
def js_divergence(left: pd.Series, right: pd.Series) -> float:
    levels = left.index.union(right.index)
    p = left.reindex(levels, fill_value=0).to_numpy(dtype=float)
    q = right.reindex(levels, fill_value=0).to_numpy(dtype=float)
    p = p / p.sum(); q = q / q.sum()
    m = 0.5 * (p + q)
    eps = 1e-12
    return float(0.5 * np.sum(p * np.log2((p + eps) / (m + eps))) + 0.5 * np.sum(q * np.log2((q + eps) / (m + eps))))

report = []
for col in ["rating", "month", "movie_id"]:
    report.append({
        "feature": col,
        "js_divergence": js_divergence(
            ratings[col].value_counts(normalize=True, dropna=False),
            synthetic[col].value_counts(normalize=True, dropna=False),
        ),
    })
pd.DataFrame(report)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ratings["rating"].value_counts(normalize=True).sort_index().plot(kind="bar", ax=axes[0], alpha=0.7, label="real")
synthetic["rating"].value_counts(normalize=True).sort_index().plot(kind="bar", ax=axes[0], alpha=0.5, label="synthetic")
axes[0].legend(); axes[0].set_title("Rating distribution")

ratings.groupby("customer_id").size().clip(upper=150).hist(ax=axes[1], bins=30, alpha=0.6, label="real")
synthetic.groupby("customer_id").size().clip(upper=150).hist(ax=axes[1], bins=30, alpha=0.5, label="synthetic")
axes[1].legend(); axes[1].set_title("History length")
plt.tight_layout()

In [ ]:
real_train, real_test = train_test_split_ratings(ratings, test_fraction=0.2, seed=SEED)
actual = real_test["rating"].to_numpy(dtype=np.float32)
models = {
    "real_sample": fit_bias_recommender(real_train, epochs=6),
    f"sdv_{SYNTHESIZER}": fit_bias_recommender(synthetic[["movie_id", "customer_id", "rating", "date"]], epochs=6),
}
pd.DataFrame([
    {"training_data": name, "rmse": rmse(actual, model.predict(real_test)), "mae": mae(actual, model.predict(real_test))}
    for name, model in models.items()
])

In [ ]:
k_summary, trials, trial_summary = evaluate_releases(
    synthetic[["movie_id", "customer_id", "rating", "date"]],
    n_known_values=(1, 2, 3),
    trials=50,
    seed=SEED,
    rare_movie_min_users=25,
    k_suppression=5,
)
trial_summary.head(12)

## Package choices

- Start with `GaussianCopulaSynthesizer`: quick, transparent, and good for a baseline.
- Try `CTGANSynthesizer` or `TVAESynthesizer`: heavier but better suited to mixed categorical/numeric tabular rows.
- For privacy guarantees, move beyond SDV's basic synthesizers toward Private-PGM/MST-style differentially private marginal synthesis.